# skyhunt — zdjęcie astronomiczne z sesji RAW (Run all)

Z folderu `MyDrive/skyhunt/raw/<SESSION>/` powstaje jedno zdjęcie nieba w pełnej rozdzielczości:
- bez ruchomych obiektów (satelity, samoloty, meteory, promienie kosmiczne: średnia z odrzucaniem σ),
- z darkami (folder z „dark” w nazwie obok, te same czasy naświetlania),
- HDR z bracketingu (jądro galaktyki z krótkich czasów),
- bez łuny i winietowania f/1.0 (model tła z pominięciem gwiazd i znanych dużych obiektów),
- z kolorem skalibrowanym na gwiazdach.

Wyniki: `MyDrive/skyhunt/out/<SESSION>/astro/` — `<SESSION>.jpg`, `<SESSION>_crop.jpg` (środek kadru), `<SESSION>_16bit.tif`, `<SESSION>_linear.fits` (do własnej obróbki, np. Siril).

Miejsce obserwacji: z `sites.yaml` na Drive (kopiowane z `COPY_SITE_FROM`, jeśli sesja nie ma własnego wpisu) — notebook nie zawiera współrzędnych.

Czas: dwa przejścia po wszystkich zdjęciach (~150 zdjęć: ~20–30 min), za pierwszym razem dochodzi master dark z folderu darków.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob

SESSION = 'm31_0310'             # folder w raw/
HINT_STAR = 'Mirach'              # jasna gwiazda w kadrze dla plate solve (Mirach jest tuż obok M31)
COPY_SITE_FROM = 's2_deneb_0210'  # skąd wziąć miejsce obserwacji, jeśli sesja nie ma wpisu w sites.yaml
DARK = ''                         # folder darków w raw/ ('' = automatycznie: pierwszy z „dark” w nazwie)
RESTACK = False                   # True = policz stos od nowa (inaczej sama obróbka zapisanego stosu, kilka minut)

REPO_URL = 'https://github.com/LukaszCiesielski-lodz/skytrack_uap.git'
BRANCH = 'main'
REPO = '/content/skytrack_uap'
DRIVE_ROOT = '/content/drive/MyDrive/skyhunt'
RAW = f'{DRIVE_ROOT}/raw'
OUT = f'{DRIVE_ROOT}/out'
CONFIG = f'{DRIVE_ROOT}/config.yaml' if os.path.exists(f'{DRIVE_ROOT}/config.yaml') else f'{REPO}/config.yaml'
print(SESSION, len(glob.glob(f'{RAW}/{SESSION}/*.[rR][aA][fF]')), 'RAF')
print('darki:', DARK or sorted(d for d in os.listdir(RAW) if 'dark' in d.lower()))

In [ ]:
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --oneline
!pip install -q -e "{REPO}[gpu,astro,report,dev,raw]"
!apt-get -qq update > /dev/null && apt-get -qq install -y astrometry.net > /dev/null && which solve-field

In [ ]:
# miejsce obserwacji i gwiazda-podpowiedź w sites.yaml (poza repo); współrzędne nie trafiają do notebooka
import yaml
SITES_FILE = f'{DRIVE_ROOT}/sites.yaml'
sites = (yaml.safe_load(open(SITES_FILE)) if os.path.exists(SITES_FILE) else None) or {}
entry = dict(sites.get(SESSION) or {})
if 'lat_deg' not in entry:
    src = sites.get(COPY_SITE_FROM) or {}
    if 'lat_deg' not in src:
        raise SystemExit(f'brak miejsca dla {SESSION} i {COPY_SITE_FROM} w sites.yaml — wpisz je w run_skyhunt.ipynb (komórka Nagrania)')
    entry.update({k: src[k] for k in ('lat_deg', 'lon_deg', 'elevation_m')})
    print(f'miejsce obserwacji skopiowane z {COPY_SITE_FROM}')
if HINT_STAR:
    entry['hint_star'] = HINT_STAR
sites[SESSION] = entry
with open(SITES_FILE, 'w') as fh:
    yaml.safe_dump(sites, fh, allow_unicode=True)
os.environ['SKYHUNT_SITES'] = SITES_FILE
print(SESSION, '→ gwiazda:', entry.get('hint_star'), '| miejsce: jest')

In [ ]:
# szybkie testy modułu (dane syntetyczne, kilka sekund)
!cd {REPO} && python -m pytest -q tests/test_astrophoto.py 2>&1 | tail -n 5

In [ ]:
LOG = f'{OUT}/skyhunt_astrophoto.log'
dark_arg = (f'--dark "{RAW}/{DARK}"' if DARK else '') + (' --restack' if RESTACK else '')
!set -o pipefail; skyhunt astrophoto "{RAW}/{SESSION}" --config "{CONFIG}" --out "{OUT}" {dark_arg} 2>&1 | tee "{LOG}" | awk -f "{REPO}/colab/dedup_log.awk"

In [ ]:
import json
from PIL import Image as PILImage
from IPython.display import Image, display

astro = f'{OUT}/{SESSION}/astro'
print(json.dumps(json.load(open(f'{astro}/astro.json')), indent=1, ensure_ascii=False))
# warianty: bazowy, Noise2Noise, Noise2Noise + dekonwolucja (wycinki ze środka), potem cały kadr
for name in (f'{SESSION}_crop.jpg', f'{SESSION}_n2n_crop.jpg', f'{SESSION}_n2n_deconv_crop.jpg',
             f'{SESSION}_n2n.jpg'):
    p = f'{astro}/{name}'
    if not os.path.exists(p):
        continue
    im = PILImage.open(p)
    im.thumbnail((1600, 1600))
    prev = f'/content/preview_{name}'
    im.save(prev, quality=90)
    print(p)
    display(Image(prev, width=1200))